In [2]:
import pandas as pd
import os
import os  # ← important pour accéder aux variables d’environnement
from dotenv import load_dotenv
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from tqdm import tqdm
df = pd.read_csv("SpotifyFeatures.csv")



In [8]:
df

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.61100,0.389,99373,0.910,0.000000,C#,0.3460,-1.828,Major,0.0525,166.969,4/4,0.814
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,0.24600,0.590,137373,0.737,0.000000,F#,0.1510,-5.559,Minor,0.0868,174.003,4/4,0.816
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,0.95200,0.663,170267,0.131,0.000000,C,0.1030,-13.879,Minor,0.0362,99.488,5/4,0.368
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.70300,0.240,152427,0.326,0.000000,C#,0.0985,-12.178,Major,0.0395,171.758,4/4,0.227
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,0.95000,0.331,82625,0.225,0.123000,F,0.2020,-21.150,Major,0.0456,140.576,4/4,0.390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232720,Soul,Slave,Son Of Slide,2XGLdVl7lGeq8ksM6Al7jT,39,0.00384,0.687,326240,0.714,0.544000,D,0.0845,-10.626,Major,0.0316,115.542,4/4,0.962
232721,Soul,Jr Thomas & The Volcanos,Burning Fire,1qWZdkBl4UVPj9lK6HuuFM,38,0.03290,0.785,282447,0.683,0.000880,E,0.2370,-6.944,Minor,0.0337,113.830,4/4,0.969
232722,Soul,Muddy Waters,(I'm Your) Hoochie Coochie Man,2ziWXUmQLrXTiYjCg2fZ2t,47,0.90100,0.517,166960,0.419,0.000000,D,0.0945,-8.282,Major,0.1480,84.135,4/4,0.813
232723,Soul,R.LUM.R,With My Words,6EFsue2YbIG4Qkq8Zr9Rir,44,0.26200,0.745,222442,0.704,0.000000,A,0.3330,-7.137,Major,0.1460,100.031,4/4,0.489


In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
import numpy as np

# Chargement du CSV de base
df = pd.read_csv("SpotifyFeatures.csv")

# Remplissage de la valeur nulle
df = df.fillna('None')

# Retire la colonne genre
df= df.drop(columns=['genre'])

# Normalise les noms d'artiste pour éviter les doublons
df['artist_name'] = df['artist_name'].str.replace('’', "'",  regex=False)  # apostrophe typographique
df['artist_name'] = df['artist_name'].str.lower().str.strip()  # minuscule + suppression espaces 

# Normalise les noms des chansons 
df['track_name'] = df['track_name'].str.replace('’', "'",  regex=False)  # apostrophe typographique
df['track_name'] = df['track_name'].str.lower().str.strip()  # minuscule + suppression espaces  

# Crée une df des doublons de track_id
df_duplicates = df[df.duplicated(subset=['track_id'])]

audio_features = ['acousticness', 'danceability', 'duration_ms', 'energy', 
                  'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 
                  'speechiness', 'tempo', 'time_signature', 'valence']

# Identifier les track_id avec des valeurs incohérentes
problematic_tracks = []

for track_id, group in df_duplicates.groupby('track_id'):
    for col in audio_features:
        if group[col].nunique() > 1:
            problematic_tracks.append(track_id)
            break  

# Supprimer ces track_id du DataFrame
df_clean = df[~df['track_id'].isin(problematic_tracks)]

# Garder la ligne avec la popularité max pour chaque track_id
df = df_clean.loc[df_clean.groupby('track_id')['popularity'].idxmax()]

# Pour éviter les doublons, nous conservons la version la plus populaire pour chaque couple track_name / artist_name.
df= df.loc[df.groupby(['track_name', 'artist_name'])['popularity'].idxmax()]

# Extraire uniquement le numérateur de la signature [2, 3, 4, 5] pour en faire une colonne numérique propre.
# Étape 1 : Extraire la partie avant le '/' et la convertir en entier
df['time_signature_clean'] = df['time_signature'].str.extract(r'^(\d+)').astype('Int64')
df['time_signature'] = df['time_signature_clean']
df = df[df['time_signature'].isin([2, 3, 4, 5])]

# Pour chaque titre de chanson unique (track_name) sélectionne la ligne ayant la popularité maximale (popularity) dans le DataFrame df
df = df.loc[df.groupby('track_name')['popularity'].idxmax()]

# On décide de ne garder que les scores au dessus de 20/100 de popularité.
df = df[df['popularity'] >= 20] 

# duration_ms entre 1min et 6min
df = df[(df['duration_ms'] >= 60000) & (df['duration_ms'] <= 360000)]

# Garde les valeurs au dessus de 0.8 sur liveness
df = df[df['liveness'] < 0.8]

# Retire les lignes qui contiennent ces motifs: "- live", "(live)", "[live]"
df = df[~df['track_name'].str.contains(r"(?:-\s*live\b|\(live\)|\[live\])", case=False, na=False, regex=True)]

# Clean en gardant les valeurs entre les bornes
df = df[(df['tempo'] >= 23.82) & (df['tempo'] <= 208.20)]

# Clean des valeurs incohérentes de loudness au dessus de 0
df = df[df['loudness'] <= 0]

# Clean des possibles podcasts/conférences au dela de 0.66 sur speechiness
df_clean = df[df['speechiness']<=0.66]

df_clean.to_csv("clean.csv", index=False)

In [15]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100731 entries, 58300 to 219463
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   artist_name           100731 non-null  object 
 1   track_name            100731 non-null  object 
 2   track_id              100731 non-null  object 
 3   popularity            100731 non-null  int64  
 4   acousticness          100731 non-null  float64
 5   danceability          100731 non-null  float64
 6   duration_ms           100731 non-null  int64  
 7   energy                100731 non-null  float64
 8   instrumentalness      100731 non-null  float64
 9   key                   100731 non-null  object 
 10  liveness              100731 non-null  float64
 11  loudness              100731 non-null  float64
 12  mode                  100731 non-null  object 
 13  speechiness           100731 non-null  float64
 14  tempo                 100731 non-null  float64
 15  t

In [16]:
pf = pd.read_csv("clean.csv")

 ## il faudra appliquer les filtres avant enrichissment de la df

In [9]:
pip install spotipy tqdm #


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Ajouter une colonne album_cover_url dans ton DataFrame df à partir des colonnes track_name et artist_name.

In [10]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")

auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)

 Fonction pour récupérer l’URL de la pochette

In [11]:
# Fonction pour récupérer l'URL de la pochette d'album à partir du titre et de l'artiste
def get_album_cover(track_name, artist_name):
    try:
        # Formate la requête de recherche compatible avec l'API Spotify
        query = f"track:{track_name} artist:{artist_name}"
        
        # Effectue la recherche via l'API Spotify, en ne gardant qu'un seul résultat
        result = sp.search(q=query, type='track', limit=1)
        
        # Extrait la liste des résultats (peut être vide)
        items = result['tracks']['items']
        
        # Si au moins un résultat est trouvé
        if items:
            # Récupère l'URL de la plus grande image de la pochette d'album
            return items[0]['album']['images'][0]['url']
        
        # Si aucun résultat trouvé
        return None
    
    # En cas d'erreur (ex : problème de connexion, résultat mal formé, etc.)
    except:
        return None


 Application sur un échantillon test (20 lignes pour commencer)

In [12]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

# Optionnel : nettoyer les doublons avant de requêter
df_test = df[['track_name', 'artist_name']].drop_duplicates().head(20).copy()

# Ajouter la colonne avec la barre de progression
df_test['album_cover_url'] = df_test.progress_apply(
    lambda row: get_album_cover(row['track_name'], row['artist_name']),
    axis=1
)

100%|██████████| 20/20 [00:02<00:00,  7.81it/s]


affiche les pochettes 

In [13]:
from IPython.display import Image, display

for url in df_test['album_cover_url'].dropna().head(5):
    display(Image(url=url))


In [14]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Fonction pour récupérer l'URL de la pochette via track_id
def get_album_cover_from_id(track_id):
    try:
        track = sp.track(track_id)  # appel direct à l'API
        images = track['album']['images']
        if len(images) >= 2:
            return images[1]['url']  # image moyenne
        elif images:
            return images[0]['url']  # fallback vers grande
        return None
    except:
        return None


In [16]:
import pandas as pd
import time
from tqdm import tqdm

# 1. Chargement de ton fichier nettoyé
df_clean = pd.read_csv("clean.csv")

# 2. On garde les 30 premiers track_id uniques pour le test
df_test = df_clean[['track_id']].drop_duplicates().head(30).copy()

# 3. Application de ta fonction get_album_cover_from_id avec tqdm
tqdm.pandas()

# 4. Traitement avec pause après le batch
print("Traitement de 30 morceaux...")
df_test['album_cover_url'] = df_test.progress_apply(
    lambda x: get_album_cover_from_id(x),
    axis=1
)

# 5. Petite pause de 4 secondes (même si ici on ne boucle pas encore)
time.sleep(4)

# 6. Fusion avec les données d'origine (pour tester le rendu final)
df_result = df_clean.merge(df_test, on='track_id', how='left')

# 7. Sauvegarde du résultat
df_result.to_csv("test_30_covers.csv", index=False)
print("✅ Fichier test_30_covers.csv enregistré.")


Traitement de 30 morceaux...


100%|██████████| 30/30 [00:00<00:00, 22067.54it/s]


✅ Fichier test_30_covers.csv enregistré.


In [17]:
# Charger le fichier de test
df_test = pd.read_csv("test_30_covers.csv")

# Aperçu des colonnes importantes
df_test[['track_name', 'artist_name', 'album_cover_url']].head(10)

,track_name,artist_name,album_cover_url
0,""" la traviata "" : amami alfredo (act ii) - dig...",maria callas,NaN
1,"""1点""",yuki hayashi,NaN
2,"""42"" - from sr3mm",rae sremmurd,NaN
3,"""45""",the gaslight anthem,NaN
4,"""99""",barns courtney,NaN
5,"""a far green country""",howard shore,NaN
6,"""all that is or ever was or ever will be""",alan silvestri,NaN
7,"""alta cagion v'aduna""",giuseppe verdi,NaN
8,"""best day ever""",alan silvestri,NaN
9,"""boom""",my first story,NaN


In [18]:
df_test['track_id'].head()


0    0wtpkz93wATDkUExJVuXEl
1    7JkDONXIbcKUQ7QzlLBumL
2    5lf91lPnGKtIqsgmG1z8Ip
3    25Sd73fleKUVPNqITPZkn1
4    6YQUuoMnRIMaOmouYoMfQr
Name: track_id, dtype: object

suite au direction équipe j'enrichie dans un premier temps ma DF avec le titre des albums 

Fonction pour récupérer le nom de l’album

In [19]:
def get_album_name_from_id(track_id):
    try:
        track = sp.track(track_id)
        return track['album']['name']
    except Exception as e:
        print(f"Erreur pour {track_id} : {e}")
        return None


In [20]:
import pandas as pd
from tqdm import tqdm

# Charger ton fichier nettoyé
df_clean = pd.read_csv("clean.csv")

# Supprimer les doublons de track_id pour éviter les requêtes inutiles
df_unique = df_clean[['track_id']].drop_duplicates().copy()

# Appliquer la fonction avec barre de progression
tqdm.pandas()
df_unique['album_name'] = df_unique['track_id'].progress_apply(get_album_name_from_id)

# Fusionner avec le DataFrame complet, sans rien supprimer
df_final = df_clean.merge(df_unique, on='track_id', how='left')

# Sauvegarder dans un nouveau fichier
df_final.to_csv("clean_with_album_name.csv", index=False)


  2%|▏         | 1637/100731 [02:50<20:26:37,  1.35it/s]

Erreur pour 0upcwsWXAgtSFAG72Ry9em : HTTPSConnectionPool(host='api.spotify.com', port=443): Read timed out. (read timeout=5)


  3%|▎         | 3418/100731 [05:46<1:59:06, 13.62it/s] WARNING:urllib3.connectionpool:Retrying (Retry(total=2, connect=None, read=False, redirect=None, status=3)) after connection broken by 'NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000015601256710>: Failed to resolve 'api.spotify.com' ([Errno 11001] getaddrinfo failed)")': /v1/tracks/7rYGZmhNJhsmVLjPidM1q8


Erreur pour 7CNxhE5lXGXSXVT1H9yBd4 : ('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))


  4%|▍         | 3921/100731 [20:22<23:15:35,  1.16it/s]

Erreur pour 6tJcc9V0oew91y00rU5EnT : HTTPSConnectionPool(host='api.spotify.com', port=443): Read timed out. (read timeout=5)


  4%|▍         | 4523/100731 [21:19<7:33:46,  3.53it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

# Fonction améliorée : affiche les erreurs et laisse le script continuer
def get_album_name_from_id(track_id):
    try:
        track = sp.track(track_id)
        return track['album']['name']
    except Exception as e:
        print(f"Erreur pour {track_id} : {e}")
        return None

# Charger le DataFrame nettoyé
df_clean = pd.read_csv("clean.csv")

# Track IDs uniques
df_unique = df_clean[['track_id']].drop_duplicates().reset_index(drop=True)

# Paramètres
batch_size = 100
results = []

# Traitement par batch
for i in tqdm(range(0, len(df_unique), batch_size)):
    batch = df_unique.iloc[i:i + batch_size].copy()
    batch['album_name'] = batch['track_id'].apply(get_album_name_from_id)
    results.append(batch)

    # Sauvegarde intermédiaire
    pd.concat(results).to_csv("temp_album_names.csv", index=False)

    # Pause pour éviter blocage
    time.sleep(5)

# Fusion finale
df_all = pd.concat(results, ignore_index=True)
df_final = df_clean.merge(df_all, on="track_id", how="left")
df_final.to_csv("clean_with_album_name.csv", index=False)
print("✅ Traitement terminé avec succès.")


  0%|          | 4/1008 [00:50<3:32:28, 12.70s/it]WARNING:root:Your application has reached a rate/request limit. Retry will occur after: 85017


In [7]:
import os

dossier = r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy"
print(os.listdir(dossier))


['.cache', '.env_moodiFy', '.git', '.gitignore', '.ipynb_checkpoints', 'analyse_spotify.ipynb', 'clean.csv', 'clean_DF.ipynb', 'csv_10000_lignes', 'df.ipynb', 'moodify_explo.py', 'pochette_enrichissement.ipynb', 'README.md', 'repartition par tempo et genre.pdf', 'SpotifyFeatures.csv', 'temp_album_names.csv', 'test_30_covers.csv']


In [10]:
import os

chemin = r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy\csv_10000_lignes"

if os.path.isfile(chemin):
    print("✅ C'est un fichier.")
elif os.path.isdir(chemin):
    print("📁 C'est un dossier.")
else:
    print("❌ Ce chemin ne correspond ni à un fichier ni à un dossier.")


📁 C'est un dossier.


In [11]:
import pandas as pd

df_10000 = pd.read_csv(r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy\csv_10000_lignes\df_test.csv")
df_10000.head()




,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,loudness,mode,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group
0,38149,paula cole,i don't want to wait,5MDQNJ7SZTytJwCbWKqJDL,62,0.2980,0.410,320027,0.468,0.000012,...,-8.177,Major,0.0553,177.508,4,0.475,4,"['female vocalists', 'singer-songwriter', 'pop']",NaN,Autres
1,38159,timmies,i dont wanna know her,4rKMHsVEAGTsA7hpKXnWrM,62,0.6020,0.835,122205,0.288,0.004610,...,-14.701,Minor,0.3410,109.934,3,0.359,3,"['Lo-Fi', 'beats', 'chill']",NaN,Autres
2,38165,death cab for cutie,i dreamt we spoke again,6TRQPx5Z0YZ0pCQX7JbtlS,62,0.2760,0.708,184827,0.706,0.006310,...,-6.450,Minor,0.0385,119.027,4,0.450,4,"['indie', 'indie rock', 'alternative']",NaN,Autres
3,10079,glass animals,black mambo,63OC8cNa4ZnFB3bbvbWCOc,62,0.2940,0.675,248920,0.409,0.000596,...,-10.707,Minor,0.0423,79.954,4,0.308,4,"['electronic', 'seen live', 'trip-hop']",NaN,Autres
4,82032,dua lipa,tears dry on their own - acoustic,6ftZrt4wcgSHokKl7ice5y,62,0.0513,0.704,196632,0.588,0.000000,...,-8.130,Minor,0.0467,125.965,4,0.708,4,['pop'],NaN,Autres


In [14]:
df_10000.columns

Index(['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity',
       'acousticness', 'danceability', 'duration_ms', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
       'speechiness', 'tempo', 'time_signature', 'valence',
       'time_signature_clean', 'genres', 'date', 'genre_group'],
      dtype='object')

In [15]:
def classer_genre(row):
    # Priorité : genre_group > genres > artist_name
    text = ''
    if pd.notna(row['genre_group']):
        text = row['genre_group'].lower()
    elif pd.notna(row['genres']):
        text = row['genres'].lower()
    elif pd.notna(row['artist_name']):
        text = row['artist_name'].lower()

    if "variété" in text or "chanson française" in text:
        return "variété française"
    elif "rap" in text or "hip hop" in text or "trap" in text:
        return "rap"
    elif "pop" in text:
        return "pop"
    elif "jazz" in text or "blues" in text:
        return "jazz/blues"
    elif "rock" in text or "indie" in text:
        return "rock/indie"
    elif "electro" in text or "dance" in text or "house" in text or "techno" in text:
        return "électro/dance"
    elif "classique" in text or "opera" in text:
        return "classique/opéra"
    else:
        return "autre"


In [16]:
df_10000["genre_simplifié"] = df_10000.apply(classer_genre, axis=1)

# Optionnel : voir la répartition
print(df_10000["genre_simplifié"].value_counts())


genre_simplifié
autre                9988
variété française      12
Name: count, dtype: int64


In [19]:
def genre_audio(row):
    if row['speechiness'] > 0.33 and row['danceability'] > 0.6 and row['valence'] < 0.5:
        return 'rap'
    elif row['danceability'] > 0.7 and row['energy'] > 0.6 and row['acousticness'] < 0.3:
        return 'électro/dance'
    elif row['acousticness'] > 0.6 and row['valence'] > 0.5:
        return 'variété française'
    elif row['instrumentalness'] > 0.8:
        return 'classique/opéra'
    elif row['energy'] < 0.4 and row['valence'] < 0.4:
        return 'jazz/blues'
    elif row['energy'] > 0.7 and row['valence'] > 0.5 and row['speechiness'] < 0.1:
        return 'rock/indie'
    else:
        return 'pop'


In [20]:
df_10000['genre_audio'] = df_10000.apply(genre_audio, axis=1)
# Optionnel : voir la répartition
print(df_10000['genre_audio'].value_counts())
# Sauvegarder le DataFrame avec les genres simplifiés
df_10000.to_csv(r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy\csv_10000_lignes\df_test_genres.csv", index=False)

genre_audio
pop                  5457
électro/dance        1883
rock/indie           1338
jazz/blues            675
variété française     374
rap                   162
classique/opéra       111
Name: count, dtype: int64


In [21]:
# Exemple avec df_10000['genres']
mask_var_fr = df_10000['genres'].str.contains(r"\b(franc|variété)\b", case=False, na=False)
mask_pas_rap = ~df_10000['genres'].str.contains(r"\b(rap|hip\s?hop)\b", case=False, na=False)

# Musiques françaises sans rap
df_variete_francaise = df_10000[mask_var_fr & mask_pas_rap]
# Sauvegarder le DataFrame filtré
df_variete_francaise.to_csv(r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy\csv_10000_lignes\df_variete_francaise.csv", index=False)

C:\Users\gaell\AppData\Local\Temp\ipykernel_9340\4281321165.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_var_fr = df_10000['genres'].str.contains(r"\b(franc|variété)\b", case=False, na=False)
C:\Users\gaell\AppData\Local\Temp\ipykernel_9340\4281321165.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pas_rap = ~df_10000['genres'].str.contains(r"\b(rap|hip\s?hop)\b", case=False, na=False)


In [22]:
mask_var_fr = df_10000['genres'].str.contains(r"\b(?:franc|variété)\b", case=False, na=False)
mask_pas_rap = ~df_10000['genres'].str.contains(r"\b(?:rap|hip\s?hop)\b", case=False, na=False)

df_variete_francaise = df_10000[mask_var_fr & mask_pas_rap]


In [23]:
df_variete_francaise.head()

,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
185,39800,céline dion,i'm alive,3qjXFLKMp4zfMmugEGPaBx,63,0.0247,0.691,210227,0.640,0.000005,...,0.0384,101.991,4,0.452,4,['variété française'],NaN,Variété Française,variété française,pop
864,85660,céline dion,the power of love - radio edit,2PFAYhBcNRj1UaluR7ZgUy,63,0.3690,0.536,286893,0.523,0.000525,...,0.0306,140.059,4,0.236,4,['variété française'],NaN,Variété Française,variété française,pop
1695,42971,louise attaque,j't'emmène au vent,0Wr98MVkENZXddiLB3bPb0,64,0.2520,0.567,184173,0.771,0.000000,...,0.0365,137.014,4,0.596,4,"['variété française', 'chanson', 'french pop',...",NaN,Variété Française,variété française,rock/indie
2433,82773,céline dion,that's the way it is,5s4catxeZsaWFnOrvrXZHf,65,0.1540,0.634,241373,0.886,0.000000,...,0.0434,93.040,4,0.577,4,['variété française'],NaN,Variété Française,variété française,rock/indie
3028,8645,céline dion,"beauty and the beast - from the soundtrack ""be...",7B3UAPLYAbwXVgbHSKEaTw,65,0.6430,0.463,249240,0.400,0.000025,...,0.0308,77.314,4,0.121,4,['variété française'],NaN,Variété Française,variété française,pop


In [25]:
# Appliquer le filtre sur la colonne 'genres'
mask_var_fr = df_10000['genres'].str.contains(r"\b(?:franc|variété)\b", case=False, na=False)
mask_pas_rap = ~df_10000['genres'].str.contains(r"\b(?:rap|hip\s?hop)\b", case=False, na=False)

# Cibler les lignes qui sont "autre" mais qui matchent le regex
mask_a_remplacer = (df_10000['genre_simplifié'] == 'autre') & mask_var_fr & mask_pas_rap

# Mise à jour
df_10000.loc[mask_a_remplacer, 'genre_simplifié'] = 'variété française'



In [26]:
df_10000["genre_simplifié"].value_counts()


genre_simplifié
autre                9988
variété française      12
Name: count, dtype: int64

In [27]:
df_10000['genre_simplifié'] = df_10000.apply(genre_audio, axis=1)


In [28]:
df_10000['genre_simplifié'].value_counts()


genre_simplifié
pop                  5457
électro/dance        1883
rock/indie           1338
jazz/blues            675
variété française     374
rap                   162
classique/opéra       111
Name: count, dtype: int64

In [29]:
for genre in df_10000['genre_simplifié'].unique():
    print(f"\n🎵 Genre : {genre.upper()}")
    display(df_10000[df_10000['genre_simplifié'] == genre].head(3))



🎵 Genre : POP


,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
0,38149,paula cole,i don't want to wait,5MDQNJ7SZTytJwCbWKqJDL,62,0.2980,0.410,320027,0.468,0.000012,...,0.0553,177.508,4,0.475,4,"['female vocalists', 'singer-songwriter', 'pop']",NaN,Autres,pop,pop
3,10079,glass animals,black mambo,63OC8cNa4ZnFB3bbvbWCOc,62,0.2940,0.675,248920,0.409,0.000596,...,0.0423,79.954,4,0.308,4,"['electronic', 'seen live', 'trip-hop']",NaN,Autres,pop,pop
4,82032,dua lipa,tears dry on their own - acoustic,6ftZrt4wcgSHokKl7ice5y,62,0.0513,0.704,196632,0.588,0.000000,...,0.0467,125.965,4,0.708,4,['pop'],NaN,Autres,pop,pop



🎵 Genre : RAP


,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
1,38159,timmies,i dont wanna know her,4rKMHsVEAGTsA7hpKXnWrM,62,0.60200,0.835,122205,0.288,0.004610,...,0.341,109.934,3,0.359,3,"['Lo-Fi', 'beats', 'chill']",NaN,Autres,rap,rap
28,36762,wiz khalifa,hot now,4Q3HHwkLVwHrSVad0RUz3V,62,0.03790,0.822,226273,0.613,0.000000,...,0.388,154.913,4,0.301,4,['rap'],NaN,Autres,rap,rap
247,32707,drake,grammys,5e574bhjycX1eH2l4Auage,63,0.00201,0.711,220427,0.441,0.000003,...,0.466,145.198,4,0.115,4,['rap'],NaN,Autres,rap,rap



🎵 Genre : ÉLECTRO/DANCE


...

,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
15,88281,sara bareilles,tightrope,1hyHgyS1V37c5gYSBmNy6R,62,0.908,0.514,218320,0.266,0.010700,...,0.0331,97.007,4,0.0647,4,"['female vocalists', 'pop', 'singer-songwriter']",NaN,Autres,jazz/blues,jazz/blues
57,81461,marcell,takkan terganti,0T4t1PywlNmJGcveGH5spB,63,0.704,0.358,241763,0.275,0.000001,...,0.0277,103.581,4,0.1170,4,['indonesian pop'],NaN,Autres,jazz/blues,jazz/blues
61,29,giacomo puccini,"""nessun dorma!""",74WjYdm3Lvbwnds4thYPUU,63,0.961,0.171,180933,0.308,0.005460,...,0.0456,171.798,5,0.0889,5,"['opera', 'classical']",NaN,Autres,jazz/blues,jazz/blues



🎵 Genre : VARIÉTÉ FRANÇAISE


,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
62,100933,hebe tien,小幸運,1ZeVIrCWzEmsJexkrgvjFv,63,0.750,0.519,265522,0.412,0.000000,...,0.0365,157.916,4,0.506,4,"['mandopop', 'c-pop', 'taiwanese pop']",NaN,Autres,variété française,variété française
75,89810,anne-marie,trigger,6xVr2J3X0zhEJQz5L0QNAP,63,0.700,0.625,193440,0.713,0.000000,...,0.0777,165.988,4,0.835,4,"['pop', 'rnb', 'seen live']",NaN,Autres,variété française,variété française
85,10757,elvis presley,blue suede shoes,47gmoUrZV3w20JAnQOZMcO,63,0.654,0.557,119200,0.660,0.000002,...,0.0560,95.252,4,0.962,4,"['rockabilly', 'rock and roll']",NaN,Autres,variété française,variété française



🎵 Genre : CLASSIQUE/OPÉRA


,Unnamed: 0,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,...,speechiness,tempo,time_signature,valence,time_signature_clean,genres,date,genre_group,genre_simplifié,genre_audio
146,80084,masego,sunday vibes,3l75jB2gKi4VpgklWIbTOz,63,0.259,0.6770,226168,0.66900,0.849,...,0.0632,170.087,4,0.2220,4,"['rnb', 'jazz', 'seen live']",NaN,Autres,classique/opéra,classique/opéra
283,41809,hugar,inngangur,4ndRB3B9iP6fx3YiCFr5es,63,0.995,0.4050,87517,0.00996,0.954,...,0.0420,67.820,3,0.3490,3,['neoclassical'],NaN,Autres,classique/opéra,classique/opéra
382,31123,brian mcbride,girl nap,542waU3uXCF7ZVyNDvePHk,63,0.953,0.0946,231213,0.18100,0.915,...,0.0361,77.092,3,0.0328,3,"['drone', 'ambient', 'neoclassical']",NaN,Autres,classique/opéra,classique/opéra


In [30]:
def genre_audio(row):
    artist = row['artist_name'].lower() if pd.notna(row['artist_name']) else ""

    artistes_variete_fr = [
        "cabrel", "goldman", "pagny", "sardou", "balavoine", "dutronc",
        "hallyday", "bashung", "mitchell", "renaud", "delpech", "fugain",
        "berger", "cloclo", "halliday", "aznavour", "bécaud", "barbelivien",
        "bruel", "calogero", "obispo", "voulzy", "souchon", "zaz",
        "christophe mae", "kendji", "vitaa", "slimane", "amel bent",
        "céline dion", "lara fabian", "francis cabrel", "louane", "jenifer",
        "m pokora", "patrick bruel", "jean-jacques goldman", "claude françois",
        "julien clerc", "juliette armanet", "vianney", "pomme", "clara luciani",
        "brigitte bardot", "sheila", "françoise hardy", "edith piaf",
        "dalida", "aline", "stone et charden", "jean ferrat", "maxime le forestier",
        "hugues aufray", "alain souchon", "alain chamfort", "jeanne cherhal",
        "aya nakamura", "indila", "marwa loud", "santa", "angèle", "anna karina", 
        "arno", "arthur h", "barbara", "bénabar", "benjamin biolay", "bernard lavilliers",
        "brigitte fontaine", "boris vian", "cali", "camille", "carla bruni",
        "catherine ringer", "christophe", "claudio capéo", "coeur de pirate",
        "coralie clément", "cyril mokaiesh", "damien saez", "daniel balavoine",
        "david guetta", "dick annegarn", "dominique a", "etienne daho",
        "france gall", "françoise hardy", "serge gainsbourg", "garou",
        "georges brassens", "georges moustaki", "gérard lenorman", "gilbert bécaud",
        "grand corps malade", "hélène ségara", "hubert-félix thiéfaine", "iam",
        "indochine", "isabelle aubret", "isabelle boulay", "jacques brel",
        "jacques dutronc", "jacques higelin", "jeanne moreau", "johnny hallyday",
        "juliette gréco", "kyo", "lâam", "laurent voulzy", "léo ferré",
        "les rita mitsouko", "louis bertignac", "m", "mc solaar",
        "maxime le forestier", "michel berger", "michel delpech", "michel jonasz",
        "michel polnareff", "mireille mathieu", "nâdiya", "natasha st-pier",
        "nekfeu", "niagara", "nolwenn leroy", "ntm", "olivia ruiz", "orelsan",
        "patricia kaas", "patrick fiori", "pauline croze", "philippe katerine",
        "pierre bachelet", "pierre perret", "pnl", "raphael", "renaud",
        "salvatore adamo", "sheila", "shy'm", "sinsemilia", "stromae",
        "superbus", "sylvie vartan", "téléphone", "thomas dutronc", "trust",
        "vanessa paradis", "william sheller", "yves duteil", "yves montand", "yseult",
        "zazie"
    ]

    # RAP
    if row['speechiness'] > 0.33 and row['danceability'] > 0.6 and row['valence'] < 0.5:
        return 'rap'

    # ÉLECTRO / DANCE
    elif row['danceability'] > 0.7 and row['energy'] > 0.6 and row['acousticness'] < 0.3:
        return 'électro/dance'

    # VARIÉTÉ FRANÇAISE - priorité sur les noms
    elif any(nom in artist for nom in artistes_variete_fr):
        return 'variété française'

    # CLASSIQUE / OPÉRA
    elif row['instrumentalness'] > 0.8:
        return 'classique/opéra'

    # JAZZ / BLUES
    elif row['energy'] < 0.4 and row['valence'] < 0.4:
        return 'jazz/blues'

    # ROCK / INDIE
    elif row['energy'] > 0.7 and row['valence'] > 0.5 and row['speechiness'] < 0.1:
        return 'rock/indie'

    # POP (par défaut)
    else:
        return 'pop'


In [31]:
from collections import Counter
genre_counts = Counter()

for _, row in df_10000.iterrows():
    genre = genre_audio(row)
    genre_counts[genre] += 1

print(dict(genre_counts))


{'pop': 4109, 'rap': 162, 'électro/dance': 1883, 'variété française': 2273, 'rock/indie': 1007, 'jazz/blues': 488, 'classique/opéra': 78}


In [32]:
import pandas as pd
import random

echantillon = df_10000[df_10000['genre_simplifié'] == 'variété française'].sample(50, random_state=42)
display(echantillon[['artist_name', 'track_name', 'genre_simplifié']])


,artist_name,track_name,genre_simplifié
8339,drake,0 to 100 / the catch up,variété française
865,dua lipa,new rules - acoustic,variété française
441,jovanny cadena y su estilo privado,aunque el mundo se oponga,variété française
8252,mac miller,what's the use?,variété française
1422,etta james,i'd rather go blind,variété française
5569,manu chao,bongo bong,variété française
1725,daryl hall & john oates,jingle bell rock - daryl's version,variété française
2598,divididos,spaghetti del rock,variété française
8439,mkto,how can i forget,variété française
2835,sza,20 something,variété française


In [33]:
def genre_audio(row):
    artist = row['artist_name'].lower().strip() if pd.notna(row['artist_name']) else ""

    artistes_variete_fr = set([
        "francis cabrel", "jean-jacques goldman", "florent pagny", "michel sardou", "daniel balavoine", "jacques dutronc",
        "johnny hallyday", "alain bashung", "eddy mitchell", "renaud", "michel delpech", "michel fugain",
        "michel berger", "claude françois", "charles aznavour", "gilbert bécaud", "didier barbelivien",
        "patrick bruel", "calogero", "pascal obispo", "laurent voulzy", "alain souchon", "zaz",
        "christophe mae", "kendji", "vitaa", "slimane", "amel bent", "céline dion", "lara fabian",
        "louane", "jenifer", "m pokora", "julien clerc", "juliette armanet", "vianney", "pomme", "clara luciani",
        "brigitte bardot", "sheila", "françoise hardy", "edith piaf", "dalida", "aline", "jean ferrat",
        "maxime le forestier", "hugues aufray", "alain chamfort", "jeanne cherhal", "aya nakamura", "indila",
        "marwa loud", "santa", "angèle", "anna karina", "arno", "arthur h", "barbara", "bénabar",
        "benjamin biolay", "bernard lavilliers", "brigitte fontaine", "boris vian", "cali", "camille",
        "carla bruni", "catherine ringer", "christophe", "claudio capéo", "coeur de pirate", "coralie clément",
        "cyril mokaiesh", "etienne daho", "france gall", "serge gainsbourg", "garou", "georges brassens",
        "georges moustaki", "gérard lenorman", "grand corps malade", "hélène ségara", "hubert-félix thiéfaine",
        "indochine", "isabelle aubret", "isabelle boulay", "jacques brel", "jacques higelin", "jeanne moreau",
        "juliette gréco", "kyo", "lâam", "laurent voulzy", "léo ferré", "les rita mitsouko", "louis bertignac",
        "m", "mc solaar", "maxime le forestier", "michel jonasz", "michel polnareff", "mireille mathieu",
        "nâdiya", "natasha st-pier", "niagara", "nolwenn leroy", "ntm", "olivia ruiz", "orelsan",
        "patricia kaas", "patrick fiori", "pauline croze", "philippe katerine", "pierre bachelet", "pierre perret",
        "pnl", "raphael", "renaud", "salvatore adamo", "shy'm", "sinsemilia", "stromae", "superbus",
        "sylvie vartan", "téléphone", "thomas dutronc", "trust", "vanessa paradis", "william sheller",
        "yves duteil", "yves montand", "yseult", "zazie"
    ])

    # RAP
    if row['speechiness'] > 0.33 and row['danceability'] > 0.6 and row['valence'] < 0.5:
        return 'rap'

    # ÉLECTRO / DANCE
    elif row['danceability'] > 0.7 and row['energy'] > 0.6 and row['acousticness'] < 0.3:
        return 'électro/dance'

    # VARIÉTÉ FRANÇAISE — match exact
    elif artist in artistes_variete_fr:
        return 'variété française'

    # CLASSIQUE / OPÉRA
    elif row['instrumentalness'] > 0.8:
        return 'classique/opéra'

    # JAZZ / BLUES
    elif row['energy'] < 0.4 and row['valence'] < 0.4:
        return 'jazz/blues'

    # ROCK / INDIE
    elif row['energy'] > 0.7 and row['valence'] > 0.5 and row['speechiness'] < 0.1:
        return 'rock/indie'

    # POP (par défaut)
    else:
        return 'pop'


In [34]:
echantillon = df_10000[df_10000['genre_simplifié'] == 'variété française'].sample(50, random_state=42)
display(echantillon[['artist_name', 'track_name', 'genre_simplifié']])

,artist_name,track_name,genre_simplifié
8339,drake,0 to 100 / the catch up,variété française
865,dua lipa,new rules - acoustic,variété française
441,jovanny cadena y su estilo privado,aunque el mundo se oponga,variété française
8252,mac miller,what's the use?,variété française
1422,etta james,i'd rather go blind,variété française
5569,manu chao,bongo bong,variété française
1725,daryl hall & john oates,jingle bell rock - daryl's version,variété française
2598,divididos,spaghetti del rock,variété française
8439,mkto,how can i forget,variété française
2835,sza,20 something,variété française


In [35]:
def genre_audio(row):
    artist = row['artist_name'].lower().strip() if pd.notna(row['artist_name']) else ""

    artistes_variete_fr = [
        "cabrel", "goldman", "pagny", "sardou", "balavoine", "dutronc",
        "hallyday", "bashung", "mitchell", "renaud", "delpech", "fugain",
        "berger", "cloclo", "aznavour", "bécaud", "barbelivien",
        "bruel", "calogero", "obispo", "voulzy", "souchon", "zaz",
        "christophe mae", "kendji", "vitaa", "slimane", "amel bent",
        "céline dion", "lara fabian", "louane", "jenifer", "m pokora",
        "julien clerc", "juliette armanet", "vianney", "pomme", "clara luciani",
        "brigitte bardot", "sheila", "françoise hardy", "edith piaf", "dalida",
        "jean ferrat", "maxime le forestier", "hugues aufray", "jeanne cherhal",
        "aya nakamura", "indila", "marwa loud", "santa", "angèle", "barbara",
        "bénabar", "benjamin biolay", "bernard lavilliers", "camille",
        "carla bruni", "catherine ringer", "christophe", "claudio capéo",
        "coeur de pirate", "cyril mokaiesh", "etienne daho", "france gall",
        "serge gainsbourg", "garou", "georges brassens", "georges moustaki",
        "gérard lenorman", "grand corps malade", "hélène ségara",
        "isabelle boulay", "jacques brel", "jacques dutronc", "juliette gréco",
        "kyo", "lâam", "laurent voulzy", "léo ferré", "les rita mitsouko",
        "louis bertignac", "mc solaar", "maxime le forestier", "michel berger",
        "michel jonasz", "michel polnareff", "mireille mathieu", "nâdiya",
        "natasha st-pier", "niagara", "nolwenn leroy", "ntm", "olivia ruiz",
        "orelsan", "patricia kaas", "patrick fiori", "pauline croze",
        "philippe katerine", "pierre bachelet", "pierre perret", "pnl",
        "raphael", "renaud", "salvatore adamo", "sheila", "shy'm",
        "sinsemilia", "stromae", "superbus", "sylvie vartan", "téléphone",
        "thomas dutronc", "trust", "vanessa paradis", "william sheller",
        "yves duteil", "yves montand", "yseult", "zazie"
    ]

    # Format intelligent de recherche
    if any(nom in artist for nom in artistes_variete_fr):
        return 'variété française'

    # Règles audio
    if row['speechiness'] > 0.33 and row['danceability'] > 0.6 and row['valence'] < 0.5:
        return 'rap'
    elif row['danceability'] > 0.7 and row['energy'] > 0.6 and row['acousticness'] < 0.3:
        return 'électro/dance'
    elif row['instrumentalness'] > 0.8:
        return 'classique/opéra'
    elif row['energy'] < 0.4 and row['valence'] < 0.4:
        return 'jazz/blues'
    elif row['energy'] > 0.7 and row['valence'] > 0.5 and row['speechiness'] < 0.1:
        return 'rock/indie'
    else:
        return 'pop'


In [36]:
# Tirer 10 morceaux au hasard dans le dataset
echantillon = df_10000.sample(10, random_state=42).copy()

# Appliquer la fonction genre_audio() sur chaque ligne de l’échantillon
echantillon["genre_testé"] = echantillon.apply(genre_audio, axis=1)

# Afficher quelques colonnes clés pour contrôle
echantillon[["artist_name", "track_name", "genre_testé"]]


,artist_name,track_name,genre_testé
6252,alkilados,ella me persigue,électro/dance
4684,nivea,don't mess with my man,électro/dance
1731,jane's addiction,jane says,pop
4742,warrant,cherry pie,pop
4521,lany,valentine's day,pop
6340,mobb deep,survival of the fittest,électro/dance
576,cody johnson,understand why,rock/indie
5202,future,goin dummi,électro/dance
6363,bts,love maze,rock/indie
439,weezer,can't knock the hustle,rock/indie


In [37]:
# Appliquer la fonction genre_audio si ce n’est pas encore fait
df_10000["genre_simplifié"] = df_10000.apply(genre_audio, axis=1)

# Tirer 50 morceaux au hasard dans la catégorie "variété française"
echantillon_variete = df_10000[df_10000["genre_simplifié"] == "variété française"].sample(50, random_state=42)

# Afficher les colonnes clés pour vérification
echantillon_variete[["artist_name", "track_name", "genre_simplifié"]]


,artist_name,track_name,genre_simplifié
2873,santaferia,haciendo nada,variété française
7899,mitchell tenpenny,drunk me,variété française
7719,aj mitchell,used to be,variété française
1941,cartel de santa,soy quien soy,variété française
7170,clara luciani,la grenade,variété française
722,gustavo santaolalla,the choice,variété française
2744,cartel de santa,el arte del engaño,variété française
8619,bass santana,make eem run!,variété française
564,cartel de santa,si estuviera en dubái,variété française
4466,indila,dernière danse,variété française


In [38]:
# Prétraitement du nom d’artiste
words_in_artist = set(artist.replace("&", " ").replace("-", " ").split())

# Intersection stricte avec les noms connus
if words_in_artist & set(artistes_variete_fr):
    return "variété française"


NameError: name 'artist' is not defined

In [39]:
artistes_variete_francaise = {
    "cabrel", "goldman", "pagny", "sardou", "balavoine", "dutronc", "hallyday", "bashung", "mitchell",
    "renaud", "delpech", "fugain", "berger", "cloclo", "aznavour", "bécaud", "barbelivien", "bruel",
    "calogero", "obispo", "voulzy", "souchon", "zaz", "christophe mae", "kendji", "vitaa", "slimane",
    "amel bent", "céline dion", "lara fabian", "louane", "jenifer", "m pokora", "julien clerc",
    "juliette armanet", "vianney", "pomme", "clara luciani", "brigitte bardot", "sheila", "françoise hardy",
    "edith piaf", "dalida", "jean ferrat", "maxime le forestier", "hugues aufray", "jeanne cherhal",
    "aya nakamura", "indila", "marwa loud", "santa", "angèle", "barbara", "bénabar", "benjamin biolay",
    "bernard lavilliers", "camille", "carla bruni", "catherine ringer", "christophe", "claudio capéo",
    "coeur de pirate", "cyril mokaiesh", "etienne daho", "france gall", "serge gainsbourg", "garou",
    "georges brassens", "georges moustaki", "gérard lenorman", "grand corps malade", "hélène ségara",
    "isabelle boulay", "jacques brel", "jacques dutronc", "juliette gréco", "kyo", "lâam",
    "laurent voulzy", "léo ferré", "les rita mitsouko", "louis bertignac", "mc solaar", "michel berger",
    "michel jonasz", "michel polnareff", "mireille mathieu", "nâdiya", "natasha st-pier", "niagara",
    "nolwenn leroy", "ntm", "olivia ruiz", "orelsan", "patricia kaas", "patrick fiori", "pauline croze",
    "philippe katerine", "pierre bachelet", "pierre perret", "pnl", "raphael", "salvatore adamo", "shy'm",
    "sinsemilia", "stromae", "superbus", "sylvie vartan", "téléphone", "thomas dutronc", "trust",
    "vanessa paradis", "william sheller", "yves duteil", "yves montand", "yseult", "zazie"
}


In [40]:
def classifier_genre(row):
    genre = row['playlist_genre'].lower()
    artist = row['artist_name'].lower().strip()

    # Vérification stricte pour la variété française
    if artist in artistes_variete_francaise:
        return 'variété française'
    
    if genre in ['pop']:
        return 'pop'
    elif genre in ['hip hop', 'rap']:
        return 'rap'
    elif genre in ['electronic', 'edm', 'dance', 'house']:
        return 'électro/dance'
    elif genre in ['rock', 'indie']:
        return 'rock/indie'
    elif genre in ['jazz', 'blues']:
        return 'jazz/blues'
    elif genre in ['classical', 'opera']:
        return 'classique/opéra'
    else:
        return 'autre'


In [41]:
print(df_10000['genre_simplifié'].value_counts())


genre_simplifié
pop                  5758
électro/dance        1871
rock/indie           1370
jazz/blues            672
rap                   162
classique/opéra       113
variété française      54
Name: count, dtype: int64


In [42]:
df_10000[df_10000['genre_simplifié'] == 'variété française'][['artist_name', 'track_name']].sample(50, random_state=42)


,artist_name,track_name
2873,santaferia,haciendo nada
7899,mitchell tenpenny,drunk me
7719,aj mitchell,used to be
1941,cartel de santa,soy quien soy
7170,clara luciani,la grenade
722,gustavo santaolalla,the choice
2744,cartel de santa,el arte del engaño
8619,bass santana,make eem run!
564,cartel de santa,si estuviera en dubái
4466,indila,dernière danse


In [47]:
import pandas as pd
import re
import unicodedata

# Chargement du fichier CSV
df = pd.read_csv(r"C:\Users\gaell\OneDrive\WILD CODE SCHOOL\Projet 2 music SPOTIFY\MoodiFy\csv_10000_lignes\df_test.csv")

# 1. Normalisation du nom de l'artiste
def normalize_text(text):
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text

df['artist_normalized'] = df['artist_name'].astype(str).apply(normalize_text)

# 2. Liste des artistes de variété française
artistes_variete = [
    "cabrel", "goldman", "pagny", "sardou", "balavoine", "dutronc",
    "hallyday", "bashung", "renaud", "delpech", "fugain", "berger",
    "cloclo", "aznavour", "becaud", "barbelivien", "bruel", "calogero",
    "obispo", "voulzy", "souchon", "zaz", "christophe mae", "kendji",
    "vitaa", "slimane", "amel bent", "celine dion", "lara fabian",
    "louane", "jenifer", "m pokora", "julien clerc", "juliette armanet",
    "vianney", "pomme", "clara luciani", "brigitte bardot", "sheila",
    "francoise hardy", "edith piaf", "dalida", "jean ferrat", "maxime le forestier",
    "hugues aufray", "jeanne cherhal", "aya nakamura", "indila", "marwa loud",
    "santa", "angele", "barbara", "benabar", "benjamin biolay", "bernard lavilliers",
    "camille", "carla bruni", "catherine ringer", "claudio capeo", "coeur de pirate",
    "cyril mokaiesh", "etienne daho", "france gall", "serge gainsbourg", "garou",
    "georges brassens", "georges moustaki", "gerard lenorman", "grand corps malade",
    "helene segara", "isabelle boulay", "jacques brel", "jacques dutronc",
    "juliette greco", "kyo", "laam", "laurent voulzy", "leo ferre",
    "les rita mitsouko", "louis bertignac", "mc solaar", "michel berger",
    "michel jonasz", "michel polnareff", "mireille mathieu", "nadiya",
    "natasha st pier", "niagara", "nolwenn leroy", "ntm", "olivia ruiz",
    "orelsan", "patricia kaas", "patrick fiori", "pauline croze", "philippe katerine",
    "pierre bachelet", "pierre perret", "pnl", "raphael", "salvatore adamo",
    "shy'm", "sinsemilia", "stromae", "superbus", "sylvie vartan",
    "telephone", "thomas dutronc", "trust", "vanessa paradis", "william sheller",
    "yves duteil", "yves montand", "yseult", "zazie"
]

# 3. Création du pattern regex
pattern = r'\b(' + '|'.join(re.escape(name) for name in artistes_variete) + r')\b'

# 4. Fonction de classification des genres
def classify_genre(row):
    artist = row['artist_normalized']
    tempo = row['tempo']
    valence = row['valence']
    danceability = row['danceability']
    speechiness = row['speechiness']
    energy = row['energy']
    instrumentalness = row['instrumentalness']

    if pd.isna(artist):
        return 'pop'  # par défaut

    if re.search(pattern, artist):
        return 'variété française'

    if speechiness > 0.35 and energy > 0.6:
        return 'rap'
    elif danceability > 0.7 and tempo > 110:
        return 'électro/dance'
    elif energy < 0.5 and danceability < 0.5:
        return 'jazz/blues'
    elif valence < 0.3 and instrumentalness > 0.4:
        return 'classique/opéra'
    elif energy > 0.6 and instrumentalness < 0.1:
        return 'rock/indie'
    else:
        return 'pop'

# 5. Application de la classification
df['genre_simplifié'] = df.apply(classify_genre, axis=1)

# 6. Comptage des genres obtenus
print(df['genre_simplifié'].value_counts())

# (Optionnel) Sauvegarde si tu veux l'utiliser ailleurs
# df.to_csv("chemin_de_sortie.csv", index=False)


genre_simplifié
rock/indie           4455
pop                  2479
électro/dance        2194
jazz/blues            604
rap                   190
classique/opéra        49
variété française      29
Name: count, dtype: int64


In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd
import time
from tqdm import tqdm

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
client_id = "5436f40cae5c4a0bb6c3d71cb05495bb",
client_secret = "2617e30595ad4023843c8899e1db7e1f",
redirect_uri = "http://127.0.0.1:8888/callback"
), requests_timeout=20)

def get_all_tracks_for_artist(artist_name):
    try:
        result = sp.search(q=f"artist:{artist_name}", type='artist', limit=1)
        items = result['artists']['items']
        if not items:
            print(f"Artiste non trouvé : {artist_name}")
            return []

        artist = items[0]
        artist_id = artist['id']

        albums = sp.artist_albums(artist_id, album_type='album,single', limit=50)
        album_ids = list({a['id'] for a in albums['items']})

        all_tracks = []
        for album_id in album_ids:
            try:
                tracks = sp.album_tracks(album_id)['items']
                for t in tracks:
                    all_tracks.append({
                        'track_id': t['id'],
                        'track_name': t['name'],
                        'artist_name': artist['name'],
                        'genre': "variété française"
                    })
            except Exception as e:
                print(f"Erreur sur album {album_id} : {e}")
            time.sleep(0.1)

        return all_tracks
    except Exception as e:
        print(f"Erreur récupération artiste : {e}")
        return []

def get_audio_features_with_popularity(track_ids):
    data = []
    for i in tqdm(range(0, len(track_ids), 50)):
        batch = track_ids[i:i+50]
        try:
            features = sp.audio_features(batch)
            details = sp.tracks(batch)['tracks']
            for feat, detail in zip(features, details):
                if feat and detail:
                    row = {
                        'track_id': feat['id'],
                        'popularity': detail['popularity'],
                        'acousticness': feat['acousticness'],
                        'danceability': feat['danceability'],
                        'duration_ms': feat['duration_ms'],
                        'energy': feat['energy'],
                        'instrumentalness': feat['instrumentalness'],
                        'key': feat['key'],
                        'liveness': feat['liveness'],
                        'loudness': feat['loudness'],
                        'mode': feat['mode'],
                        'speechiness': feat['speechiness'],
                        'tempo': feat['tempo'],
                        'time_signature': feat['time_signature'],
                        'valence': feat['valence']
                    }
                    data.append(row)
        except Exception as e:
            print(f"Erreur audio features : {e}")
        time.sleep(0.3)
    return pd.DataFrame(data)

def main():
    french_artists = [
        "Francis Cabrel", "Jean-Jacques Goldman", "Florent Pagny", "Michel Sardou",
        "Daniel Balavoine", "Jacques Dutronc", "Johnny Hallyday", "Alain Bashung",
        "Renaud", "Michel Delpech", "Michel Fugain", "Serge Gainsbourg",
        "Mylène Farmer", "Patrick Bruel", "Vanessa Paradis", "Julien Clerc",
        "Véronique Sanson", "Charles Aznavour", "Claude François", "Édith Piaf",
        "Benjamin Biolay", "Zazie", "Christophe", "Camille", "Cali", "Carla Bruni",
        "Maxime Le Forestier", "Gérard Lenorman", "Francis Lalanne", "Louis Chedid",
        "Yves Duteil", "Léo Ferré", "Laurent Voulzy", "Philippe Chatel",
        "Pierre Bachelet", "Isabelle Boulay", "Charles Trenet", "Julien Doré",
        "M. Pokora", "Coeur de Pirate", "France Gall", "Françoise Hardy",
        "Alain Souchon", "Nolwenn Leroy", "Alizée", "Yseult", "Aya Nakamura",
        "Stromae", "Indochine", "Angèle", "Téléphone", "Les Rita Mitsouko",
        "Pascal Obispo", "Michel Polnareff", "Jean Ferrat", "Hugues Aufray",
        "Georges Brassens", "Jacques Brel", "Claude Nougaro", "Serge Lama",
        "Michel Jonasz", "Émile & Images", "Daniel Guichard", "Miossec", "Kyo",
        "Superbus", "Gérard Manset", "Alain Chamfort", "Alain Barrière",
        "Nino Ferrer", "Gilbert Bécaud", "Sheila", "Dalida", "Mireille Mathieu",
        "Nathalie Cardone", "Pierre Perret", "Daniel Lévi", "Liane Foly",
        "Amel Bent", "Indila", "Kendji Girac", "Vianney", "Pomme",
        "Clara Luciani", "Juliette Armanet", "Louane", "Jenifer", "Arthur H",
        "Calogero", "Shy'm", "Sinsemilia", "Vitaa", "Slimane", "Zaho",
        "L'Impératrice", "Lou-Adriane Cassidy", "Malajube", "Klô Pelgag",
        "Les Louanges", "Marie-Pierre Arthur", "Charlotte Gainsbourg",
        "Jane Birkin", "Étienne Daho", "Phoenix", "Air", "Gaël Faye",
        "Polo & Pan", "Christine and the Queens", "Loïc Nottet", "GIMS",
        "Black M", "Wejdene", "La Femme", "Keren Ann", "Yael Naim",
        "Noémi Waysfeld", "Bernard Lavilliers", "Grand Corps Malade",
        "MC Solaar", "IAM", "NTM", "Jul", "Daft Punk", "Yelle",
        "Matmatah", "Louise Attaque", "Soldat Louis", "Gilbert Montagné",
        "La Bande à Basile", "Les Wampas", "Hélène Rollès", "Axelle Red",
        "Suzane", "Tété", "Cats on Trees", "Les Innocents", "Nolwenn Leroy",
        "Raphaël", "Olivia Ruiz", "Thomas Dutronc", "Nolwenn Korbell",
        "Daphné", "Dominique A", "Pauline Croze", "Camille Lellouche",
        "Cléa Vincent", "Féfé", "Sébastien Tellier", "Alex Beaupain",
        "Bertrand Belin", "Barcella", "Bénabar", "Renan Luce", "Manu Chao",
        "Tryo", "Zebda", "Sinik", "Diam's", "Lomepal", "Orelsan",
        "Eddy de Pretto", "Hoshi", "Terrenoire", "Léa Paci", "Tim Dup",
        "Fishbach", "Clarika", "Emily Loizeau", "Barbara", "Maurane",
        "Frida Boccara", "Linda Lemay", "Jean-Louis Aubert", "Jacques Higelin",
        "Jean-Jacques Debout", "Roch Voisine", "Enrico Macias",
        "Les Stentors", "Jean-Luc Lahaye", "Julie Zenatti", "Marie Myriam",
        "Chimène Badi", "Lara Fabian", "Hélène Ségara", "Elsa Lunghini",
        "Jeanne Mas", "Patricia Kaas", "Nicole Croisille", "Nicoletta",
        "Stone et Charden", "Peter et Sloane", "Boris Vian", "Dick Annegarn",
        "Brigitte Fontaine", "Mano Solo", "Jean Guidoni", "Coralie Clément",
        "Dominique Dalcan", "Marc Lavoine", "Daniel Lavoie", "Jean-Patrick Capdevielle",
        "Vincent Delerm", "Hervé Vilard", "Dave", "Frédéric François"
    ]

    print("Collecte des morceaux...")
    all_tracks = []
    for name in tqdm(french_artists):
        all_tracks.extend(get_all_tracks_for_artist(name))

    print(f"{len(all_tracks)} morceaux récupérés. Suppression doublons...")
    df_tracks = pd.DataFrame(all_tracks).drop_duplicates(subset='track_id')
    df_tracks = df_tracks[df_tracks['track_id'].notnull()]

    print("Récupération des audio features + popularité...")
    df_features = get_audio_features_with_popularity(df_tracks['track_id'].tolist())

    print("Fusion des données...")
    df_final = df_tracks.merge(df_features, on='track_id', how='inner')

    # Nettoyage pour aucune valeur nulle
    df_final.dropna(inplace=True)

    # On peut aussi retirer les titres sans popularité
    df_final = df_final[df_final['popularity'] > 0]

    # Vérification stricte
    assert df_final.isnull().sum().sum() == 0, "Des valeurs nulles sont encore présentes !"

    # Réorganisation colonnes
    df_final = df_final[[
        'genre', 'artist_name', 'track_name', 'track_id', 'popularity',
        'acousticness', 'danceability', 'duration_ms', 'energy',
        'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
        'speechiness', 'tempo', 'time_signature', 'valence'
    ]]

    print(f"{len(df_final)} morceaux valides avec toutes les données complètes.")

    print("Sauvegarde vers CSV...")
    df_final.to_csv('df_variete_francaise_complete.csv', index=False)
    print("Fichier enregistré : df_variete_francaise_complete.csv")

if __name__ == "__main__":
    main()

Collecte des morceaux...


  0%|          | 1/205 [00:14<49:53, 14.68s/it]

Erreur récupération artiste : HTTPSConnectionPool(host='accounts.spotify.com', port=443): Max retries exceeded with url: /api/token (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002180888CC20>: Failed to resolve 'accounts.spotify.com' ([Errno 11001] getaddrinfo failed)"))


 35%|███▍      | 71/205 [07:52<14:48,  6.63s/it]

Erreur sur album 5Wvnwde3Z4e0zEjcIBsDSP : ('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))


 82%|████████▏ | 168/205 [52:21<02:16,  3.70s/it]   WARNING:root:Your application has reached a rate/request limit. Retry will occur after: 83269
